# PE NPU 멀티코어 처리량(throughput) 벤치마크

Mobilint ARIES NPU에서 **여러 코어를 동시에 활용해 처리량을 올리는** 공식 방법을 측정하는 노트북.

## Mobilint 코어 모드 (컴파일 시 지정, `docs/multicore.md`)

| 모드 | 동작 | 용도 |
|------|------|------|
| **Single** | 8개 Local Core가 각자 독립 추론 | 코어에 **쉴 새 없이** 요청을 던지면(async/멀티스레딩) throughput 최대 |
| **Multi** | 4 코어(1 Cluster)가 협력해 **4-batch** 처리 | 통상적 배치 처리. 멀티스레딩 없이 4배치 throughput |
| **Global4/8** | 4/8 코어가 **한 데이터**를 분담 | latency(단건 지연) 감소, 무거운 모델 |

### 핵심: "배치를 모델 입력에 박는" 게 아니다
NPU는 batch 차원을 입력에 넣는 대신 **코어 모드 + 비동기 요청**으로 배치/처리량을 낸다.
우리 `pe_full.mxq`는 **Single 모드**로 컴파일돼 있고(입력 `[336,336,3]` 단건), 8코어에 async로
요청을 흘려보내면 순차 대비 큰 throughput을 얻는다. (Multi/Global을 쓰려면 그 모드로 **재컴파일** 필요 — 맨 아래 참고)

> 측정 대상은 **NPU trunk**(무거운 24 transformer block)다. attn_pool head는 CPU(float)라 별도이며 연산량 1% 미만.

## 0. 환경 + MXQ 코어 모드 확인

In [ ]:
import os, sys, time, threading
sys.path.insert(0, os.path.abspath(".."))
import numpy as np
import qbruntime

MXQ = "../pe_npu/out/pe_full.mxq"
assert os.path.exists(MXQ), f"MXQ 없음: {MXQ} (먼저 컴파일)"
assert os.path.exists("/dev/aries0"), "NPU 디바이스(/dev/aries0) 없음"

# 이 MXQ가 어떤 코어 모드로 컴파일됐는지 + 할당 코어 확인
_acc = qbruntime.Accelerator(); _m = qbruntime.Model(MXQ); _m.launch(_acc)
print("MXQ 코어 모드   :", _m.get_core_mode())
print("입력 shape      :", _m.get_model_input_shape())
print("할당된 코어 수  :", len(_m.get_target_cores()), "개")
_m.dispose()

## 1. 측정 준비
단건 입력(HWC `(336,336,3)`)과 측정 횟수. 각 측정은 warmup 1회 후 N회 평균.
실제 이미지가 아니어도 처리량(시간)은 동일하므로 더미 입력으로 측정한다.

In [ ]:
N = 40                                              # 측정 횟수
x = np.random.rand(336, 336, 3).astype(np.float32)  # 단건 HWC 입력

def img_per_s(elapsed, n=N): return n / elapsed
print(f"측정 횟수 N={N}, 입력 {x.shape} {x.dtype}")

## 2. 동기(순차) — baseline
한 장 추론 -> 결과 대기 -> 다음 장. CPU가 매번 NPU를 기다려서 **사실상 코어 1개만** 쓰인다.
단건 latency의 기준값이기도 하다.

In [ ]:
acc = qbruntime.Accelerator()
m = qbruntime.Model(MXQ); m.launch(acc)
m.infer(x)  # warmup

t = time.perf_counter()
for _ in range(N):
    m.infer(x)
sync = time.perf_counter() - t
m.dispose()

lat_ms = sync / N * 1000
print(f"[동기] 총 {sync*1000:.0f}ms | 장당 latency {lat_ms:.1f}ms | {img_per_s(sync):.1f} img/s")

## 3. async 파이프라인 (권장) — 8코어 동시 활용
`set_async_pipeline_enabled(True)` 후 `infer_async()`로 요청을 **밀어 넣고** 나중에 `future.get()`으로 회수.
런타임이 빈 코어에 자동 분배해 8코어를 쉴 새 없이 돌린다. Mobilint가 권장하는 single-mode throughput 방식.
(`docs/advanced_usage.md`)

In [ ]:
mc = qbruntime.ModelConfig(); mc.set_async_pipeline_enabled(True)
acc_a = qbruntime.Accelerator()
ma = qbruntime.Model(MXQ, mc); ma.launch(acc_a)
ma.infer_async(x).get()  # warmup

t = time.perf_counter()
futures = [ma.infer_async(x) for _ in range(N)]   # 비동기로 전부 제출
results = [f.get() for f in futures]              # 결과 회수
asyn = time.perf_counter() - t
ma.dispose()

print(f"[async] 총 {asyn*1000:.0f}ms | {img_per_s(asyn):.1f} img/s | 동기 대비 speedup x{sync/asyn:.2f}")

## 4. (대안) 멀티스레딩
async가 어려운 코드 구조면, 동기 `infer`를 여러 스레드에서 동시 호출해도 8코어가 분배된다.
qbruntime의 동기 `infer`는 thread-safe하므로 ThreadPool로 던지면 된다.

In [ ]:
from concurrent.futures import ThreadPoolExecutor

acc_t = qbruntime.Accelerator()
mt = qbruntime.Model(MXQ); mt.launch(acc_t)
mt.infer(x)  # warmup

t = time.perf_counter()
with ThreadPoolExecutor(max_workers=8) as ex:
    list(ex.map(lambda _: mt.infer(x), range(N)))
thr = time.perf_counter() - t
mt.dispose()

print(f"[threading x8] 총 {thr*1000:.0f}ms | {img_per_s(thr):.1f} img/s | 동기 대비 x{sync/thr:.2f}")

## 5. 결과 비교

In [ ]:
import matplotlib.pyplot as plt

labels = ["sync\n(sequential)", "async\n(pipeline)", "threading\n(x8)"]
ips    = [img_per_s(sync), img_per_s(asyn), img_per_s(thr)]
colors = ["#9aa0a6", "#2a9d8f", "#457b9d"]

fig, ax = plt.subplots(figsize=(6,4))
bars = ax.bar(labels, ips, color=colors)
ax.set_ylabel("Throughput (img/s)")
ax.set_title(f"NPU trunk throughput (Single mode, N={N})")
for b, v in zip(bars, ips):
    ax.text(b.get_x()+b.get_width()/2, v+max(ips)*0.01, f"{v:.1f}", ha="center", fontsize=10)
plt.tight_layout(); plt.show()

print(f"동기 {ips[0]:.1f} -> async {ips[1]:.1f} img/s (x{ips[1]/ips[0]:.1f})  : 8코어 동시 활용 효과")

## 6. 전체(full NPU) 처리량 + Multi/Global 모드

### full NPU 전체
실제 서비스는 image→embedding 전부 NPU head(`MXQInferenceFull`)다. CPU head는 작지만(연산 1% 미만)
동기 루프 구조라, 대량 처리 시엔 NPU는 async, CPU head는 별도 스레드로 파이프라인하면 위 throughput을 거의 그대로 살릴 수 있다.

### Multi / Global 모드를 쓰려면 (재컴파일 필요)
현재 MXQ는 **Single 전용**이라 multi/global을 못 쓴다. 코어 모드는 컴파일 시 고정이기 때문.
- `pe_npu/compile.py`의 `inference_scheme="single"`을 `"multi"`(4-batch) / `"global4"`/`"global8"`(단건 latency↓)로 바꿔 재컴파일
- 런타임에서 `mc.set_multi_core_mode(...)` / `set_global8_core_mode()` 등으로 실행

| 목표 | 선택 |
|------|------|
| 대량 일괄(throughput) | **Single + async**(지금 이 노트북) 또는 **Multi**(4-batch 재컴파일) |
| 단건 지연 최소(latency) | **Global4/8** 재컴파일 |
| 실시간 단건 1장 | 현재 Single 동기로 충분 |

어떤 모드가 유리한지는 워크로드(단건 vs 대량)에 따라 다르니, 필요하면 multi/global로 재컴파일해 이 노트북으로 다시 측정하면 된다.